# InternImage — Vision Foundation Models with Deformable Convolutions (DCNv3)

**Paper:** Wang, W., Dai, J., Chen, Z., Huang, Z., Li, Z., Zhu, X., Hu, X., Lu, T., Lu, L., Li, H., Wang, X., & Qiao, Y. (2023). InternImage: Exploring Large-Scale Vision Foundation Models with Deformable Convolutions. *CVPR 2023*. https://arxiv.org/abs/2211.05778

**Implementation:** `hale_core/nn/vision/cnn/internimage.py`

> InternImage proposes DCNv3 — a separable reformulation of deformable convolution analogous to the ViT attention mechanism — enabling scalable pretraining of large CNN foundation models (up to 3B parameters) that match or exceed ViT-scale transformers.

## Architecture Overview

```
InternImage Backbone:

Input [B, 3, H, W]
  → Stem: 4×4 stride-4 Conv + LayerNorm             [B, C, H/4, W/4]
  → Stage 0: N0 × InternImageBlock                  [B, C,   H/4,  W/4]
  → Downsample (3×3 stride-2 + LN)                  [B, 2C,  H/8,  W/8]
  → Stage 1: N1 × InternImageBlock                  [B, 2C,  H/8,  W/8]
  → Downsample                                       [B, 4C,  H/16, W/16]
  → Stage 2: N2 × InternImageBlock                  [B, 4C,  H/16, W/16]
  → Downsample                                       [B, 8C,  H/32, W/32]
  → Stage 3: N3 × InternImageBlock                  [B, 8C,  H/32, W/32]
  → GlobalAvgPool → [B, 8C]

InternImageBlock (pre-norm Transformer-style):
  x → LayerNorm → DCNv3 → (+x) → LayerNorm → FFN → (+x)

DCNv3 (Separable Deformable Conv):
  vs DCNv2 (O(C²) per group): each group has own projection
  vs DCNv3 (O(C) total):      shared depthwise projection, separate offset/mask per group

DCNv3 Forward:
  x [B,H,W,C] → dw_conv (depthwise BN+GELU, shared)     → x' [B,H,W,C]
             → offset_conv (C → G×K×2 offsets)           → Δp [B,H,W,G,K,2]
             → mask_conv   (C → G×K masks, softmax)      → m  [B,H,W,G,K]
             → grid_sample(x', base+Δp) per group g       → sampled [B,G,K,H,W,C//G]
             → x_sampled * m → proj (C → C)              → output [B,H,W,C]
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(17, 12))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.axis('off')

def box(ax, x, y, w, h, text, sub='', fc='#1f6feb', fs=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle='round,pad=0.07',
                        facecolor=fc, edgecolor='#58a6ff', lw=1.3)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2+(0.1 if sub else 0), text, ha='center', va='center',
            color='white', fontsize=fs, fontweight='bold', fontfamily='monospace')
    if sub:
        ax.text(x+w/2, y+h/2-0.18, sub, ha='center', va='center',
                color='#8b949e', fontsize=6.5)

def arr(ax, x1, y1, x2, y2, c='#58a6ff'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=c, lw=1.4))

# ── LEFT: DCNv3 vs DCNv2 comparison ─────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 11)
ax.set_ylim(0, 14)
ax.text(5.5, 13.5, 'DCNv2 vs DCNv3 Architecture', ha='center',
        color='#f0f6fc', fontsize=11, fontweight='bold')

ax.text(2.5, 13.0, 'DCNv2 (per-group projection)', ha='center',
        color='#f85149', fontsize=9, fontweight='bold')
ax.text(8.0, 13.0, 'DCNv3 (shared depthwise + separate offsets)', ha='center',
        color='#2ea043', fontsize=9, fontweight='bold')
ax.axvline(5.5, color='#30363d', lw=1.5, linestyle=':')

# DCNv2
box(ax, 0.5, 11.5, 4.5, 0.8, 'Input x [B,C,H,W]', '', fc='#21262d', fs=8)
for g in range(3):
    gy = 9.5 - g*0.1
    box(ax, 0.3+g*0.2, gy, 4.0-g*0.2, 1.2, f'Group {g+1}:\nfc_offset+fc_proj (O(C²))',
        '', fc='#da3633', fs=7)
    arr(ax, 2.75, 11.5, 2.75-g*0.1, gy+1.2)
ax.text(2.5, 9.2, '...G groups, each O(C²)', ha='center', color='#8b949e', fontsize=7.5)
box(ax, 0.5, 7.8, 4.5, 0.8, 'Sum + Output [B,C,H,W]', '', fc='#21262d', fs=8)
arr(ax, 2.75, 9.5, 2.75, 8.6)
ax.text(2.5, 7.3, 'Total cost: O(G × C²)', ha='center', color='#f85149', fontsize=8.5)

# DCNv3
box(ax, 5.8, 11.5, 4.8, 0.8, 'Input x [B,H,W,C]', '', fc='#21262d', fs=8)

box(ax, 5.8, 10.2, 4.8, 0.9, 'DW pre-projection (shared)',
    'dw_conv: C→C, BN, GELU  O(C)', fc='#1f6feb', fs=7.5)
arr(ax, 8.2, 11.5, 8.2, 11.1)

# branch to offsets and masks
box(ax, 5.8, 8.8, 2.2, 0.8, 'offset_conv', 'C→G×K×2', fc='#6e40c9', fs=7.5)
box(ax, 8.4, 8.8, 2.2, 0.8, 'mask_conv', 'C→G×K\n(softmax)', fc='#6e40c9', fs=7.5)
arr(ax, 8.2, 10.2, 7.0, 9.6)
arr(ax, 8.2, 10.2, 9.5, 9.6)

box(ax, 5.8, 7.5, 4.8, 0.9, 'Grid Sample per group',
    'F.grid_sample(x\'[g], coords[g]) → sampled', fc='#2ea043', fs=7.5)
arr(ax, 7.0, 8.8, 8.2, 8.4)
arr(ax, 9.5, 8.8, 8.2, 8.4)

box(ax, 5.8, 6.2, 4.8, 0.9, 'Weight by masks → proj',
    'sampled × m → Linear(C, C)', fc='#388bfd', fs=7.5)
arr(ax, 8.2, 7.5, 8.2, 7.1)

box(ax, 5.8, 5.2, 4.8, 0.7, 'Output [B,H,W,C]', '', fc='#21262d', fs=8)
arr(ax, 8.2, 6.2, 8.2, 5.9)
ax.text(8.2, 4.8, 'Total cost: O(G × C)', ha='center', color='#2ea043', fontsize=8.5)
ax.text(8.2, 4.4, '(G × C² in DCNv2 → G × C in DCNv3)', ha='center', color='#8b949e', fontsize=8)

# Bottom comparison table
ax.text(5.5, 3.8, '─────── Key Differences ───────', ha='center', color='#8b949e', fontsize=8)
rows = [
    ('Projection',  'Per-group O(C²)',    'Shared DW O(C)'),
    ('Mask norm',   'Sigmoid (DCNv2)',     'Softmax (DCNv3)'),
    ('Memory',      'High (G×C² each)',   'Low (C total)'),
    ('Scale',       'Max ~1B params',      'Scales to 3B+'),
]
for i, (prop, v2, v3) in enumerate(rows):
    ax.text(0.5, 3.3 - i*0.55, f'{prop}:', color='#f0f6fc', fontsize=8)
    ax.text(3.5, 3.3 - i*0.55, v2, color='#f85149', fontsize=8)
    ax.text(6.5, 3.3 - i*0.55, v3, color='#2ea043', fontsize=8)

# ── RIGHT: InternImageBlock detail ───────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 14)
ax.text(5, 13.5, 'InternImageBlock', ha='center', color='#f0f6fc',
        fontsize=12, fontweight='bold')

box(ax, 2.5, 12.3, 5, 0.8, 'Input x', '[B, H, W, C] (channels-last)', fc='#21262d', fs=8)

box(ax, 2.5, 11.0, 5, 0.9, 'LayerNorm', 'pre-norm on input', fc='#388bfd')
arr(ax, 5, 12.3, 5, 11.9)
box(ax, 2.5, 9.5,  5, 1.2, 'DCNv3', 'DW pre-proj→offset/mask→\ngrid_sample→weight→proj', fc='#1f6feb', fs=8)
arr(ax, 5, 11.0, 5, 10.7)
box(ax, 2.5, 8.3,  5, 0.9, 'Add (+x)', 'first residual', fc='#2ea043')
arr(ax, 5, 9.5, 5, 9.2)
ax.annotate('', xy=(3.0, 8.5), xytext=(1.5, 12.3),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=2.0,
                            connectionstyle='arc3,rad=0.0'))
arr(ax, 5, 8.3, 5, 8.0)

box(ax, 2.5, 7.1,  5, 0.9, 'LayerNorm', 'pre-norm before FFN', fc='#388bfd')
arr(ax, 5, 8.3, 5, 8.0)
box(ax, 2.5, 5.9,  5, 0.9, 'FFN (4× expand)', 'Linear→GELU→Dropout→Linear', fc='#1f6feb', fs=8)
arr(ax, 5, 7.1, 5, 6.8)
box(ax, 2.5, 4.7,  5, 0.9, 'Add (+x)', 'second residual', fc='#2ea043')
arr(ax, 5, 5.9, 5, 5.6)
ax.annotate('', xy=(3.0, 4.9), xytext=(1.5, 8.3),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=2.0,
                            connectionstyle='arc3,rad=0.0'))
arr(ax, 5, 4.7, 5, 4.4)
box(ax, 2.5, 3.5,  5, 0.7, 'Output', '[B, H, W, C]', fc='#21262d')

# variant table
ax.text(5, 3.0, 'Registered Variants', ha='center', color='#f0f6fc', fontsize=9, fontweight='bold')
ax.text(5, 2.5, 'Variant       C    depths           groups', ha='center',
        color='#58a6ff', fontsize=7.5, fontfamily='monospace')
rows = [
    'tiny   64  [4,4,18,4]   [4,8,16,32]',
    'small  80  [4,4,21,4]   [5,10,20,40]',
    'base   112 [4,4,21,4]   [7,14,28,56]',
    'large  160 [5,5,22,5]   [10,20,40,80]',
    'huge   320 [6,6,32,6]   [20,40,80,160]',
]
for i, r in enumerate(rows):
    ax.text(5, 2.0 - i*0.45, r, ha='center', color='#8b949e',
            fontsize=7.5, fontfamily='monospace')

plt.suptitle('InternImage / DCNv3 Architecture', fontsize=14, color='#f0f6fc',
             fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## Key Innovations

### 1. The CNN Scaling Problem
ViT-style transformers (ViT, Swin, DeiT) achieved strong results at large scales by leveraging global self-attention. Could CNNs also scale to foundation-model size while maintaining their inductive biases (translation equivariance, spatial locality)? The challenge: DCNv2's per-group projection is O(G × C²), making it too expensive to scale.

### 2. DCNv3 — Separable Deformable Convolution
DCNv3 redesigns deformable convolution as a separable operation:
- **Shared depthwise pre-projection**: A single depthwise conv processes all channels (O(C) cost). This is *shared* across all G groups.
- **Per-group offset prediction**: Each group has its own lightweight offset predictor for K sampling points. The G groups learn diverse receptive fields.
- **Softmax masking** (DCNv3) vs sigmoid masking (DCNv2): Softmax normalises across the K sampling points, ensuring the weights sum to 1. This improves stability and interpretability.

**Analogy to ViT attention**:  
ViT: Q×K (O(N²C)) + V projection (O(C²)) per head  
DCNv3: offset prediction (O(GKC)) + shared DW (O(C)) per group  
Both use multi-group/head factorisation to reduce cost.

### 3. InternImageBlock — Pre-Norm Transformer Style
The block follows the ViT post-norm-free design:
```
x = x + DCNv3(LN(x))   # spatial mixing (like attention)
x = x + FFN(LN(x))     # channel mixing (like MLP)
```
This structure allows stacking many blocks without gradient vanishing, mirrors ViT's architecture, and enables joint pre-training with transformers.

### 4. Adaptive Receptive Field
Unlike fixed-grid convolutions, DCNv3 learns WHERE to look. For each output position, each group learns K offset points: the network can sample the car's wheels and roof for a 'car' feature, or adjacent characters for a text feature. Different groups learn complementary sampling patterns.

### 5. Pure PyTorch Implementation (No CUDA Extension)
The official InternImage requires a compiled CUDA extension (`dcnv3_ops`). This implementation uses `F.grid_sample` which is a standard PyTorch op with bilinear interpolation, providing identical mathematical behaviour with full autograd support — at the cost of slightly lower throughput vs the CUDA kernel.

## Component Walkthrough

### `DCNv3.__init__` (lines 96–140 in `internimage.py`)
```python
# Shared depthwise pre-projection (O(C) — not per-group)
self.dw_conv = nn.Sequential(
    nn.Conv2d(channels, channels, dw_kernel, groups=channels, bias=False),  # depthwise
    nn.BatchNorm2d(channels),
    nn.GELU(),
)
# Per-group offset: G × K × 2 channels total
self.offset_conv = nn.Linear(channels, groups * self.K * 2)
nn.init.zeros_(self.offset_conv.weight)  # start as regular grid
# Per-group mask: G × K channels, softmax normalised across K
self.mask_conv   = nn.Linear(channels, groups * self.K)
# Output projection
self.proj = nn.Linear(channels, channels)
```

### `DCNv3._deformable_sample` (lines 195–285)
```python
# Create base sampling grid [1, Hs, Ws, 1, 1, 2]
grid_base = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).unsqueeze(3).unsqueeze(4)
# Regular k×k kernel offsets [1, 1, 1, 1, K, 2]
base_k_norm = base_k.view(1, 1, 1, 1, K, 2)  # K grid points, normalised
# Learned offsets [B, Hs, Ws, G, K, 2]
# grid_base + base_k_norm broadcasts to [1, Hs, Ws, 1, K, 2]
# + off_norm [B, Hs, Ws, G, K, 2] → [B, Hs, Ws, G, K, 2]
coords = (grid_base + base_k_norm + off_norm).clamp(-1, 1)

# For each group g, sample input at group-specific offsets
for g in range(G):
    cg = coords[:, :, :, g, :, :].contiguous()  # [B, Hs, Ws, K, 2]
    cg = cg.permute(0, 3, 1, 2, 4).reshape(B*K, Hs, Ws, 2)  # [B*K, Hs, Ws, 2]
    xg = x_groups[:, g].unsqueeze(1).expand(-1, K, -1, -1, -1).reshape(B*K, Cg, H, W)
    out_g = F.grid_sample(xg, cg, mode='bilinear', align_corners=False)  # [B*K, Cg, Hs, Ws]
    # reshape → [B, Cg, Hs, Ws, K] and accumulate
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

x = torch.randn(1, 3, 224, 224)

for name in ['internimage_tiny', 'internimage_small', 'internimage_base']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    with torch.no_grad():
        out = bb(x)
    print(f'{name:20s}  embed={out.pooled_output.shape[-1]}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params: {bb.num_parameters():,}')
    print()


## References

1. **[Primary]** Wang, W., et al. (2023). **InternImage: Exploring Large-Scale Vision Foundation Models with Deformable Convolutions**. *CVPR 2023*. https://arxiv.org/abs/2211.05778

2. Zhu, X., et al. (2019). **Deformable ConvNets v2 (DCNv2)**. *CVPR 2019*. https://arxiv.org/abs/1811.11168 — DCNv2 that DCNv3 improves upon.

3. Dai, J., et al. (2017). **Deformable Convolutional Networks (DCNv1)**. *ICCV 2017*. https://arxiv.org/abs/1703.06211 — Original deformable convolution.

4. Liu, Z., et al. (2021). **Swin Transformer**. *ICCV 2021*. — Transformer architecture that InternImage's block structure mirrors (pre-norm, [attention+FFN] structure → [DCNv3+FFN]).

5. He, K., et al. (2022). **Masked Autoencoders Are Scalable Vision Learners**. *CVPR 2022*. — MAE pretraining framework adapted for InternImage foundation model training.